In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/panther_merged.csv")

cols_to_drop = ['occupants', 'numberoffloors', 'rating', 'source_eui', 'site_eui',
                 'subindustry', 'date_opened', 'energystarscore', 'steam', 'hotwater',
                 'heatingtype', 'solar', 'industry', 'precipDepth6HR', 'leed_level',
                 'irrigation', 'chilledwater', 'gas']
df = df.drop(columns=cols_to_drop)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['building_id', 'timestamp']).reset_index(drop=True)

df.shape

C:\Users\HP\AppData\Local\Temp\ipykernel_19952\1872787775.py:4: DtypeWarning: Columns (0: chilledwater, 1: water, 2: irrigation, 3: gas, 4: eui, 5: leed_level) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/panther_merged.csv")


(1842120, 24)

In [2]:
eui_col = pd.read_csv("../data/processed/panther_merged.csv", usecols=['eui'], dtype=str)
df['eui'] = pd.to_numeric(eui_col['eui'], errors='coerce')
df.shape

(1842120, 24)

In [4]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

df[['timestamp', 'hour', 'day_of_week', 'month', 'is_weekend']].head()

,timestamp,hour,day_of_week,month,is_weekend
0,2016-01-01 00:00:00,0,4,1,0
1,2016-01-01 01:00:00,1,4,1,0
2,2016-01-01 02:00:00,2,4,1,0
3,2016-01-01 03:00:00,3,4,1,0
4,2016-01-01 04:00:00,4,4,1,0


In [5]:
df['lag_1h'] = df.groupby('building_id')['meter_reading'].shift(1)
df['lag_24h'] = df.groupby('building_id')['meter_reading'].shift(24)
df['lag_168h'] = df.groupby('building_id')['meter_reading'].shift(168)

df[['building_id', 'timestamp', 'meter_reading', 'lag_1h', 'lag_24h', 'lag_168h']].head(30)

,building_id,timestamp,meter_reading,lag_1h,lag_24h,lag_168h
0,Panther_assembly_Carrol,2016-01-01 00:00:00,NaN,NaN,NaN,NaN
1,Panther_assembly_Carrol,2016-01-01 01:00:00,NaN,NaN,NaN,NaN
2,Panther_assembly_Carrol,2016-01-01 02:00:00,NaN,NaN,NaN,NaN
3,Panther_assembly_Carrol,2016-01-01 03:00:00,NaN,NaN,NaN,NaN
4,Panther_assembly_Carrol,2016-01-01 04:00:00,NaN,NaN,NaN,NaN
5,Panther_assembly_Carrol,2016-01-01 05:00:00,NaN,NaN,NaN,NaN
6,Panther_assembly_Carrol,2016-01-01 06:00:00,NaN,NaN,NaN,NaN
7,Panther_assembly_Carrol,2016-01-01 07:00:00,NaN,NaN,NaN,NaN
8,Panther_assembly_Carrol,2016-01-01 08:00:00,NaN,NaN,NaN,NaN
9,Panther_assembly_Carrol,2016-01-01 09:00:00,NaN,NaN,NaN,NaN


In [6]:
df[df['building_id'] == 'Panther_education_Rosalie'][['building_id', 'timestamp', 'meter_reading', 'lag_1h', 'lag_24h', 'lag_168h']].dropna(subset=['meter_reading']).head(30)

,building_id,timestamp,meter_reading,lag_1h,lag_24h,lag_168h
456848,Panther_education_Rosalie,2016-01-30 08:00:00,12.8025,NaN,NaN,NaN
456869,Panther_education_Rosalie,2016-01-31 05:00:00,11.0021,NaN,NaN,NaN
456881,Panther_education_Rosalie,2016-01-31 17:00:00,15.4030,NaN,NaN,NaN
458510,Panther_education_Rosalie,2016-04-08 14:00:00,17.4033,NaN,NaN,NaN
459067,Panther_education_Rosalie,2016-05-01 19:00:00,131.2958,NaN,NaN,NaN
459522,Panther_education_Rosalie,2016-05-20 18:00:00,68.2132,NaN,NaN,NaN
459523,Panther_education_Rosalie,2016-05-20 19:00:00,81.4157,68.2132,NaN,NaN
459524,Panther_education_Rosalie,2016-05-20 20:00:00,70.0135,81.4157,NaN,NaN
459525,Panther_education_Rosalie,2016-05-20 21:00:00,70.6136,70.0135,NaN,NaN
459526,Panther_education_Rosalie,2016-05-20 22:00:00,74.0143,70.6136,NaN,NaN


In [7]:
df['rolling_24h_mean'] = df.groupby('building_id')['meter_reading'].transform(lambda x: x.rolling(window=24, min_periods=1).mean())

df[df['building_id'] == 'Panther_education_Rosalie'][['timestamp', 'meter_reading', 'rolling_24h_mean']].dropna(subset=['meter_reading']).tail(10)

,timestamp,meter_reading,rolling_24h_mean
473678,2017-12-31 14:00:00,60.2116,56.352550
473679,2017-12-31 15:00:00,60.0116,56.502579
473680,2017-12-31 16:00:00,58.6113,56.544254
473681,2017-12-31 17:00:00,58.2112,56.535917
473682,2017-12-31 18:00:00,56.8110,57.061021
473683,2017-12-31 19:00:00,47.8092,56.986004
473684,2017-12-31 20:00:00,45.0087,56.769296
473685,2017-12-31 21:00:00,44.6086,56.652604
473686,2017-12-31 22:00:00,45.8088,56.444229
473687,2017-12-31 23:00:00,45.4088,56.002479


In [8]:
df['rolling_168h_mean'] = df.groupby('building_id')['meter_reading'].transform(lambda x: x.rolling(window=168, min_periods=1).mean())
df.shape

(1842120, 33)

In [9]:
df[['lag_1h', 'lag_24h', 'lag_168h']].isna().mean()

lag_1h      0.198179
lag_24h     0.199452
lag_168h    0.207370
dtype: float64

In [10]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/panther_features.csv", index=False)

In [12]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/panther_features.csv", index=False)